# Measured feel

PLAN.md is explicit: no hand-authored swing parameter. Timing is something we
**measure** off the trained model and report, not something we set.

"The model rushes closed hats by 8 ms" is a finding. A swing slider would be a
guess dressed up as a feature.

This notebook loads a checkpoint, runs the validation split, and reports per
class: mean deviation from the metric grid, spread, measured swing ratio, and
the hat-vs-kick offset.

In [ ]:
import sys, pathlib
import numpy as np, torch

ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import feel, train as train_mod
from build import device_of, get_subgraph, load_config
from realtime import load_checkpoint

CHECKPOINT = ROOT / 'runs' / 'v1_8piece' / 'best.pt'
model, kit, cfg = load_checkpoint(CHECKPOINT, torch.device('cpu'))
print(kit.classes)

In [ ]:
_, val_loader, _ = train_mod.build_loaders(cfg, kit)
step_ms = cfg['audio']['step_ms']

reports = []
with torch.no_grad():
    for wav, y, style, tempo in val_loader:
        logits, _ = model(wav)
        prob = torch.sigmoid(logits.float()).numpy()
        ref = y.numpy()
        n = min(prob.shape[1], ref.shape[1])
        for i in range(prob.shape[0]):
            t = float(tempo[i])
            if t > 0:
                reports.append(feel.analyse(prob[i, :n], ref[i, :n], kit.classes, step_ms, t))

print(feel.aggregate(reports).to_text())

## Reading the table

- **grid dev ms** — positive is behind the beat, negative is rushing. A
  consistent non-zero value with a *small* spread is a pocket, not an error.
- **spread ms** — how tight the placement is. This is what the `tightness`
  slider (global GABA gain) moves.
- **swing** — 1.0 is dead straight, ~1.5 is triplet swing. Measured from the
  model's own output.
- **vs ref ms** — offset against the human performance, for matched onsets only.

Compare against the ablation arms: if the real connectome and a degree-matched
rewiring produce the same timing signature, the topology is not shaping feel.

In [ ]:
# Slider sweep: does pocket actually drag the beat?
from build import role_index
roles = role_index(get_subgraph(cfg))

for value in (0.7, 1.0, 1.4):
    model.reset_sliders()
    model.set_slider('pocket', value, roles)
    rs = []
    with torch.no_grad():
        for wav, y, style, tempo in val_loader:
            logits, _ = model(wav)
            prob = torch.sigmoid(logits.float()).numpy(); ref = y.numpy()
            n = min(prob.shape[1], ref.shape[1])
            for i in range(prob.shape[0]):
                t = float(tempo[i])
                if t > 0:
                    rs.append(feel.analyse(prob[i, :n], ref[i, :n], kit.classes, step_ms, t))
    agg = feel.aggregate(rs)
    print(f"pocket={value}: grid dev {agg.overall['grid_dev_ms']:+.1f} ms, "
          f"spread {agg.overall['grid_spread_ms']:.1f} ms")
model.reset_sliders()